# Taller: Clasificadores Bayesianos con `digits`

**Evaluación 3 — Inteligencia Artificial**

En este cuaderno se construye y evalúa un clasificador bayesiano para los dígitos escritos del conjunto `digits` de `scikit-learn`. Se cumple la metodología solicitada:

- 80 % de los datos para entrenamiento y 20 % para prueba.
- Validación cruzada estratificada con `K-fold = 4` únicamente sobre el conjunto de entrenamiento.
- Comparación de varios modelos bayesianos.
- Selección del modelo más adecuado según la naturaleza de los datos y su desempeño.
- Aplicación del suavizado de Laplace y análisis del efecto de `alpha`.

> **Nota metodológica:** el conjunto de prueba se reserva hasta el final. Así, la validación cruzada sirve para seleccionar el modelo y el conjunto de prueba ofrece una estimación final no utilizada durante la selección.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42


## 1. Carga y exploración del conjunto de datos

El conjunto `digits` contiene imágenes de 8 × 8 píxeles. Cada imagen se representa mediante 64 atributos y cada atributo es un nivel de intensidad entre 0 y 16. La variable objetivo tiene diez clases: los dígitos del 0 al 9.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"Número de observaciones: {X.shape[0]}")
print(f"Número de atributos: {X.shape[1]}")
print(f"Clases: {np.unique(y)}")
print(f"Rango de intensidades: [{X.min():.0f}, {X.max():.0f}]")
print("Distribución de clases:")
print(pd.Series(y).value_counts().sort_index().to_string())

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.ravel()):
    first_index = np.flatnonzero(y == digit)[0]
    ax.imshow(digits.images[first_index], cmap="gray_r")
    ax.set_title(f"Clase {digit}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Partición 80/20 y configuración de la validación cruzada

Se realiza una partición estratificada para conservar aproximadamente la misma proporción de cada dígito en entrenamiento y prueba. Después, se define `StratifiedKFold(n_splits=4)` para que cada pliegue conserve también la distribución de clases. La semilla fija permite reproducir los resultados.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

print(f"Entrenamiento: {X_train.shape[0]} observaciones ({X_train.shape[0] / len(X):.1%})")
print(f"Prueba: {X_test.shape[0]} observaciones ({X_test.shape[0] / len(X):.1%})")
print(f"Número de pliegues de validación cruzada: {cv.n_splits}")


## 3. Comparación de clasificadores bayesianos

Se comparan tres variantes de Naive Bayes:

1. **GaussianNB:** supone que cada atributo sigue una distribución normal (gaussiana) condicionada a la clase.
2. **MultinomialNB:** modela conteos o intensidades no negativas; es razonable para píxeles que representan cantidades de intensidad.
3. **BernoulliNB:** convierte los atributos en eventos binarios; resulta apropiado cuando solo importa si un píxel está activo o no.

La métrica principal para elegir el modelo será el `F1-macro`, porque trata por igual a las diez clases. También se reportan exactitud, precisión y recall.

In [ ]:
modelos = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB (alpha=1)": MultinomialNB(alpha=1.0),
    "BernoulliNB": BernoulliNB(alpha=1.0, binarize=0.0),
}

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
}

filas = []
for nombre, modelo in modelos.items():
    resultados_cv = cross_validate(
        modelo, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1
    )
    filas.append({
        "Modelo": nombre,
        "Accuracy CV": resultados_cv["test_accuracy"].mean(),
        "Precision macro CV": resultados_cv["test_precision_macro"].mean(),
        "Recall macro CV": resultados_cv["test_recall_macro"].mean(),
        "F1 macro CV": resultados_cv["test_f1_macro"].mean(),
        "Desv. F1 CV": resultados_cv["test_f1_macro"].std(),
    })

comparacion = pd.DataFrame(filas).sort_values("F1 macro CV", ascending=False)
comparacion.reset_index(drop=True)


### Justificación de la distribución elegida

Los atributos no son variables binarias: son intensidades enteras no negativas entre 0 y 16, donde un valor mayor representa mayor presencia de tinta en el píxel. Por ello, una distribución **multinomial** es una hipótesis natural para representar la masa de intensidad de cada imagen. Además, `MultinomialNB` permite aplicar directamente suavizado de Laplace mediante `alpha`.

Aunque `GaussianNB` puede ser competitivo, su supuesto de normalidad por píxel no describe exactamente la naturaleza acotada y discreta de las intensidades. `BernoulliNB` pierde información al binarizar los niveles de gris. En consecuencia, se selecciona el modelo con mayor `F1 macro` promedio en la validación cruzada; en caso de que las diferencias sean pequeñas, la interpretación de los atributos y la posibilidad de suavizado favorecen a `MultinomialNB`.

## 4. Efecto del suavizado de Laplace

En un modelo multinomial, el suavizado de Laplace reemplaza la estimación de probabilidad condicional por:

$$P(x_j\mid y=c)=\frac{N_{cj}+\alpha}{N_c+\alpha d}$$

donde `N_cj` es la cantidad observada para el atributo `j` en la clase `c`, `N_c` es el total de la clase, `d` es el número de atributos y `alpha` controla la intensidad del suavizado. Con `alpha=1` se tiene el suavizado de Laplace clásico. Un valor positivo evita probabilidades cero y hace que el modelo sea menos sensible a atributos ausentes en una clase.

Se prueban varios valores de `alpha` usando los mismos cuatro pliegues. La selección del valor se realiza sin tocar el conjunto de prueba.

In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
filas_alpha = []

for alpha in alphas:
    modelo = MultinomialNB(alpha=alpha)
    resultados_cv = cross_validate(
        modelo, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1
    )
    filas_alpha.append({
        "alpha": alpha,
        "Accuracy CV": resultados_cv["test_accuracy"].mean(),
        "Precision macro CV": resultados_cv["test_precision_macro"].mean(),
        "Recall macro CV": resultados_cv["test_recall_macro"].mean(),
        "F1 macro CV": resultados_cv["test_f1_macro"].mean(),
        "Desv. F1 CV": resultados_cv["test_f1_macro"].std(),
    })

tabla_alpha = pd.DataFrame(filas_alpha).sort_values("F1 macro CV", ascending=False)
display(tabla_alpha.style.format({
    "Accuracy CV": "{:.4f}",
    "Precision macro CV": "{:.4f}",
    "Recall macro CV": "{:.4f}",
    "F1 macro CV": "{:.4f}",
    "Desv. F1 CV": "{:.4f}",
}))

alpha_optimo = tabla_alpha.iloc[0]["alpha"]
print(f"Alpha seleccionado por F1 macro promedio: {alpha_optimo}")
print("El suavizado de Laplace clásico corresponde a alpha=1.0.")


In [ ]:
alpha_plot = pd.DataFrame(filas_alpha).sort_values("alpha")
plt.figure(figsize=(8, 4.5))
plt.semilogx(alpha_plot["alpha"], alpha_plot["Accuracy CV"], "o-", label="Accuracy")
plt.semilogx(alpha_plot["alpha"], alpha_plot["F1 macro CV"], "s-", label="F1 macro")
plt.axvline(1.0, color="gray", linestyle="--", label="Laplace: alpha=1")
plt.xlabel("alpha (escala logarítmica)")
plt.ylabel("Promedio en validación cruzada")
plt.title("Efecto de alpha en MultinomialNB")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


### Interpretación del efecto de `alpha`

- Cuando `alpha` es muy pequeño, el modelo se acerca a la estimación sin suavizado y puede asignar probabilidades extremadamente pequeñas a combinaciones poco observadas.
- Al aumentar `alpha`, las probabilidades se alejan de cero y el modelo gana estabilidad, especialmente para clases o píxeles con poca evidencia.
- Si `alpha` es demasiado grande, todas las probabilidades se vuelven más parecidas entre sí: aparece subajuste y pueden disminuir `accuracy` y `F1`.
- Por tanto, el mejor valor se determina con los cuatro pliegues del entrenamiento. `alpha=1` es la elección clásica de Laplace, mientras que el valor óptimo puede ser distinto porque se selecciona según el desempeño de este conjunto concreto.

## 5. Evaluación final sobre el conjunto de prueba

Se entrena `MultinomialNB` con el `alpha` seleccionado y con todos los datos de entrenamiento. Solo ahora se evalúa sobre el 20 % de prueba reservado.

In [ ]:
modelo_final = MultinomialNB(alpha=float(alpha_optimo))
modelo_final.fit(X_train, y_train)
y_pred = modelo_final.predict(X_test)

metricas_test = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision macro", "Recall macro", "F1 macro"],
    "Valor": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="macro", zero_division=0),
        recall_score(y_test, y_pred, average="macro", zero_division=0),
        f1_score(y_test, y_pred, average="macro", zero_division=0),
    ],
})
display(metricas_test.style.format({"Valor": "{:.4f}"}))

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues", values_format="d", ax=ax
)
ax.set_title(f"Matriz de confusión — MultinomialNB (alpha={alpha_optimo})")
plt.tight_layout()
plt.show()


## Conclusiones

1. La división estratificada 80/20 mantiene el balance entre las diez clases y la prueba permanece aislada hasta la evaluación final.
2. La validación cruzada estratificada de cuatro pliegues permite comparar modelos con menor dependencia de una sola partición del entrenamiento.
3. `MultinomialNB` es una elección interpretable para `digits` porque sus atributos son intensidades no negativas y discretas; conserva la información de niveles de gris que perdería un modelo Bernoulli.
4. El suavizado de Laplace evita probabilidades condicionales cero. `alpha` debe ser positivo, pero un valor excesivo produce sobre-suavizado y puede reducir el desempeño.
5. El resultado más importante es la métrica obtenida en el conjunto de prueba después de seleccionar el modelo y `alpha` sin utilizar ese conjunto durante el proceso de decisión.